In [ ]:
import numpy as np

from pennylane.core import Operator
from tests.templates.subroutines.arithmetic.test_signed_out_multiplier import twos_complement_value

lst = np.array([[1 / 2, 0], [0.2, 1 / 3]])
lst[0, :]

In [ ]:
import pennylane as qp


@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.X(1)
    qp.Toffoli([0, 1, 2])
    qp.ops.op_math.Controlled(qp.X(2), control_wires=[0, 1], control_values=[0, 0])


qp.draw(circuit)()

In [ ]:
import pennylane


@qp.qnode(qp.device("default.qubit"))
def circuit():
    qp.OutMultiplier([0], [1], [2])


qp.draw(circuit, level="device")()

In [ ]:
num_wires = 10
for k in range(num_wires - 3, -1, -1):
    print(k)

In [ ]:
wires = [0, 2, 4, 6]
work_wires = [1, 3, 5]
all_wires = wires[:1] + list(sum(zip(wires[1:], work_wires), start=tuple()))
all_wires

In [ ]:
from sympy.ntheory import factorint

num_wires = 8
f, _ = tuple(factorint(num_wires).items())[0]
permutation = [(i // f) + (i % f) * (num_wires // f) for i in range(num_wires)]
permutation

In [ ]:
num_wires = 12
permutation = [0 for _ in range(num_wires)]
for i in range(num_wires // 2):
    permutation[i] = i * 2
k = num_wires - 1
for j in range(len(wires) // 2):
    permutation[k] = k - j
    k -= 1
permutation

In [ ]:
from scipy.constants import physical_constants

physical_constants["Bohr radius"][0] * 1e10

In [ ]:
import pennylane as qp

qp.layer

In [ ]:
import pennylane as qp

dev = qp.device("default.qubit")


@qp.decompose(gate_set=[qp.Elbow, qp.CNOT])
@qp.qnode(dev)
def circuit():
    qp.MultiControlledX([6, 5, 4, 3, 2, 1, 0], [1, 1, 1, 1, 1, 1])
    qp.MultiControlledX([5, 4, 3, 2, 1, 0], [1, 1, 1, 1, 1])
    qp.MultiControlledX([4, 3, 2, 1, 0], [1, 1, 1, 1])
    qp.MultiControlledX([3, 2, 1, 0], [1, 1, 1])
    qp.MultiControlledX([2, 1, 0], [1, 1])
    qp.MultiControlledX([1, 0], [1])
    return qp.state()


qp.draw(circuit)()

In [ ]:
from pennylane.resource import specs
from pennylane.templates import TemporaryAND, Elbow
from pennylane.decomposition import enable_graph
from pennylane.ops import Controlled, CNOT, PauliX, GlobalPhase, adjoint
from pennylane import device, Incrementer, state, qnode, decompose, inspect_decomps
import pennylane as qp

dev = device("default.qubit")

enable_graph()


@decompose(gate_set={TemporaryAND: 1, CNOT: 1, "Adjoint(TemporaryAND)": 1})
@qnode(dev)
def circuit(wires, work_wires):
    Controlled(
        Incrementer(wires, work_wires),
        [12],
        [1],
    )
    return state()


print(qp.draw(circuit, level=1)([0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]))

print(
    inspect_decomps(
        Controlled(
            Incrementer([0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]),
            [12],
            [1],
        )
    )
)

print(specs(circuit)([0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]))

In [ ]:
from pennylane.measurements import sample
from pennylane.templates import BasisEmbedding
from pennylane import OutAdder, device, qnode
from tests.templates.subroutines.arithmetic.test_signed_out_multiplier import twos_complement_value

dev = device("default.qubit", wires=[0, 1, 2, 3, 4, 5, 6])


@qnode(dev, shots=1)
def circ():
    BasisEmbedding([1, 1, 1, 0], [0, 1, 2, 3])
    OutAdder([0, 1], [2, 3], [4, 5])
    return sample(wires=[4, 5])


res = circ()
val = twos_complement_value(res[0])
val

In [ ]:
for elem in [0, 1, 2, 3]:
    if elem == 1:
        del elem
        continue
    print(elem)

In [ ]:
from jax import lax
from jax import numpy as jnp
from pennylane.control_flow import for_loop

target_wires = []
i = 5
wires = jnp.array([1, 2, 3])
num_wires = 3


@for_loop(i)
def build_wires(j, nw, tw, w):
    tw.append(lax.dynamic_index_in_dim(w, j + (nw - i), keepdims=False))
    return nw, tw, w


target_wires = build_wires(num_wires, target_wires, wires)
target_wires

In [ ]:
from collections.abc import Iterable

control_wires = [0.3]
isinstance(control_wires, (str, int)) or (
    isinstance(control_wires, Iterable) and isinstance(control_wires[0], (str, int))
)

In [ ]:
import jax
import pennylane as qp


def f(w1, w2):
    qp.ctrl(qp.X(w1), control=w2, control_values=(1,))


qp.capture.enable()

f(jax.numpy.array(2), jax.numpy.array(3))

In [ ]:
from pennylane.templates.subroutines.ffft import FFFT
from pennylane import device, draw, qnode, state, draw_mpl

dev = device("default.qubit", wires=(0, 1, 2, 3, 4, 5, 6, 7))

wires = (0, 1, 2, 3, 4, 5, 6, 7)


@qnode(dev)
def circuit(wires):
    FFFT(wires, True)
    return state()


# print(draw(circuit, level="device")(wires))
draw_mpl(circuit, level="device", style="pennylane_sketch")(wires)

In [ ]:
import pennylane as qp
from pennylane import SignedOutMultiplier, state

dev = qp.device("default.qubit")


@qp.qnode(dev)
def multiply(x_wires, y_wires, output_wires, work_wires):
    SignedOutMultiplier(x_wires, y_wires, output_wires, work_wires)
    return state()


x_wires, y_wires, output_wires, work_wires = (
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8, 9),
    (10, 11, 12, 13, 14, 15),
)

print(qp.draw(multiply, level="device")(x_wires, y_wires, output_wires, work_wires))

In [ ]:
from pennylane.templates.subroutines.arithmetic.incrementer import (
    _controlled_incrementer_decomposition,
)
import pennylane as qp
import numpy as np
from pennylane import Incrementer, state, sample
from pennylane.ops import Controlled, PauliX

dev = qp.device("default.qubit")

wires, work_wires = (0, 1, 2, 3, 4, 5), (6, 7, 8, 9, 10, 11, 12)

incr = Incrementer(wires, work_wires)


@qp.qnode(dev, shots=1)
def inc(wires):
    # PauliX(12)
    # _controlled_incrementer_decomposition(
    #     control_wires=[12],
    #     work_wires=[],
    #     base=incr,
    # )
    Incrementer(wires, work_wires)
    return sample(wires=wires)


print(qp.draw(inc, level="device")(wires))
print(inc(wires))

In [ ]:
from pennylane.templates.subroutines.arithmetic.incrementer import (
    _controlled_incrementer_decomposition,
)
import pennylane as qp
import numpy as np
from pennylane import Incrementer, state, sample
from pennylane.ops import Controlled, PauliX

dev = qp.device("default.qubit")

wires, work_wires = (0, 1, 2, 3, 4, 5), (6, 7, 8, 9, 10, 11, 12)

incr = Incrementer(wires, work_wires)


@qp.qnode(dev, shots=1)
def inc(wires):
    _controlled_incrementer_decomposition(
        control_wires=[12],
        work_wires=[],
        base=incr,
    )
    return sample(wires=wires)


print(qp.draw(inc, level="device")(wires))
print(inc(wires))

In [ ]:
import pennylane as qp
import numpy as np

dev = qp.device("default.qubit", wires=4)
addrs = ["000", "001"]
amps = np.array([np.sqrt(0.3), np.sqrt(0.7)])

shots = 1000


@qp.set_shots(shots)
@qp.qnode(dev, interface=None)
def circuit():
    qp.FFQRAM(amplitudes=amps, wires=[0, 1, 2, 3], address=addrs)
    return qp.counts()


results = circuit()

# Post-select on register qubit = 1.
filtered_states = []
filtered_counts = []

for basis_state, count in results.items():
    if basis_state[-1] == "1":
        filtered_states.append(basis_state)
        filtered_counts.append(count)

# Normalize to conditional probabilities.
filtered_counts = np.array(filtered_counts)
p_reg1 = filtered_counts.sum() / shots
conditioned_probs = filtered_counts / shots / p_reg1
obtained_amplitudes = np.sqrt(conditioned_probs)

print("expected success probability:", 1 / (2 ** len(addrs[0])))
print("obtained success probability:", p_reg1)

print("expected states:", [addr + "1" for addr in addrs])
print("obtained states:", filtered_states)

print("expected amplitudes:", amps)
print("obtained amplitudes:", obtained_amplitudes)

In [ ]:
import pennylane as qp

x = -3
y = 3

x_wires = [0, 1, 2]
y_wires = [3, 4, 5]
output_wires = [6, 7, 8, 9, 10, 11]
work_wires = [12, 13, 14, 15]

dev = qp.device("default.qubit")


@qp.qnode(dev, shots=1)
def circuit():
    qp.BasisEmbedding(x, wires=x_wires)
    qp.BasisEmbedding(y, wires=y_wires)
    qp.SignedOutMultiplier(x_wires, y_wires, output_wires, work_wires, output_wires_zeroed=True)
    return qp.sample(wires=output_wires)


print(qp.draw(circuit, level=1)())

In [ ]:
import pennylane as qp

dev = qp.device("default.qubit")


@qp.qnode(dev)
def iqp_circuit(weights, pattern, spin_sym):
    qp.IQP(weights, [0, 1], pattern, spin_sym=spin_sym)
    return [qp.expval(qp.PauliZ(0)), qp.expval(qp.PauliZ(1))]


iqp_circuit([0.89, 0.54], [[[0]], [[1]]], spin_sym=False)

In [ ]:
from pennylane.workflow import construct_tape
import pennylane as qml
from pennylane import qjit
from pennylane.decomposition import enable_graph

# the compile pipeline we want to replace our QFTs *before* they get to Catalyst
pipeline = qml.CompilePipeline(
    qml.decompose(gate_set=qml.decomposition.gate_sets.CLIFFORD_T_PLUS_RZ)
)

# shared setup
enable_graph()
dev = qml.device("null.qubit", wires=(0, 1, 2))


# This example works as expected
@qml.qnode(dev)
def circuit():
    qml.QFT((0, 1, 2))
    return qml.expval(qml.Z(1))


decomposed = pipeline(circuit)
qjited = qjit(decomposed, capture=True)
qjited.jit_compile(tuple())


# This example fails, as the QFT is still in the circuit by the time we get to
# the DecompRuleInterpreter in Catalyst, which cannot handle it and errors out
@qml.qnode(dev)
def circuit():
    qml.adjoint(qml.QFT)((0, 1, 2))
    return qml.expval(qml.Z(1))


decomposed = pipeline(circuit)

tape = construct_tape(decomposed, level="user")()
print(tape.operations)

qjited = qjit(decomposed, capture=True)
qjited.jit_compile(tuple())

In [ ]:
import pennylane.estimator as qre

print("explicit precision control")
control = qre.GQSPTimeEvolution(qre.RX(0.1, wires=0), 1.0, 1.0, 0.01)
print(qre.estimate(control))

print("default precision trigger")
op = qre.GQSPTimeEvolution(qre.RX(0.1, wires=0), 1.0, 1.0)
print("stored poly_approx_precision:", op.poly_approx_precision)
print(qre.estimate(op))

In [ ]:
import pennylane.estimator as qre

print("eta=2 control")
control = qre.FirstQuantization(1000, 2, omega=100.0)
print("lamb:", control.lamb)
print("gates:", control.gates)
print("qubits:", control.qubits)

print("eta=1 trigger")
trigger = qre.FirstQuantization(1000, 1, omega=100.0)
print("lamb:", trigger.lamb)
print("gates:", trigger.gates)
print("qubits:", trigger.qubits)

In [ ]:
import pennylane as qp

dev = qp.device("default.qubit")


@qp.qnode(dev)
def iqp_circuit(weights, pattern, spin_sym):
    qp.IQP(weights, 2, pattern, spin_sym=spin_sym)
    return [qp.expval(qp.PauliZ(0)), qp.expval(qp.PauliZ(1))]


iqp_circuit([0.89, 0.54], [[[0]], [[1]]], spin_sym=False)

In [ ]:
# Direct `label` reproducer:

import numpy as np
import pennylane as qml

op = qml.s_prod(np.array([1.234, 2.345]), qml.X(0))

print("matrix shape:", qml.matrix(op).shape)
print("label:", op.label(decimals=2))

# `qml.draw` reproducer:


import numpy as np
import pennylane as qml

op = qml.s_prod(2.345, qml.X(0))
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def circuit():
    return qml.expval(op)


print(qml.draw(circuit)())

In [ ]:
import numpy as np
import pennylane as qml

op = qml.pow(qml.Z(0), 0.5)

print("qml.eigvals:", qml.eigvals(op))
print("matrix eigvals:", np.linalg.eigvals(qml.matrix(op)))

In [ ]:
import numpy as np
import pennylane as qml


def random_su2(rng):
    X = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
    Q, R = np.linalg.qr(X)
    d = np.diag(R)
    Q = Q * (d / np.abs(d))
    Q /= np.linalg.det(Q) ** 0.5
    return Q


def compose_ops(ops):
    U = np.eye(4, dtype=complex)
    for op in ops:
        U = qml.matrix(op, wire_order=(0, 1)) @ U
    return U


def phase_align_error(U, V):
    overlap = np.vdot(V.reshape(-1), U.reshape(-1))
    phase = overlap / abs(overlap)
    return np.linalg.norm(U - phase * V)


rng = np.random.default_rng(123)
A, B, C, D = [random_su2(rng) for _ in range(4)]

ops = [
    qml.QubitUnitary(A, 0),
    qml.QubitUnitary(B, 1),
    qml.CNOT((1, 0)),
    qml.RZ(8.471910266820082e-04, 0),
    qml.RX(-1.3556395803158893e-03, 1),
    qml.CNOT((1, 0)),
    qml.QubitUnitary(C, 0),
    qml.QubitUnitary(D, 1),
]

U = compose_ops(ops)
decomp = qml.ops.two_qubit_decomposition(U, wires=(0, 1))
U2 = compose_ops(decomp)

print("returned ops:", [op.name for op in decomp])
print("num CNOTs:", sum(op.name == "CNOT" for op in decomp))
print("phase-aligned error:", phase_align_error(U, U2))

In [ ]:
import pennylane as qml
import numpy as np

print("qml.is_commuting(CZ, SWAP):", qml.is_commuting(qml.CZ(wires=[0, 1]), qml.SWAP(wires=[0, 1])))

A = qml.matrix(qml.CZ(wires=[0, 1]))
B = qml.matrix(qml.SWAP(wires=[0, 1]))
print("AB == BA:", np.allclose(A @ B, B @ A))

In [ ]:
import pennylane as qml

m0 = qml.measure(0)

# --- Step 1: Two opposite predicates from the same measurement ---
# "measurement outcome equals 0" vs "measurement outcome equals 1"
expr_eq_0 = m0 == 0
expr_eq_1 = m0 == 1

# Their branches prove they are semantically opposite:
print("(m0==0).branches:", expr_eq_0.branches)  # {(0,): True, (1,): False}
print("(m0==1).branches:", expr_eq_1.branches)  # {(0,): False, (1,): True}

# But qml.equal says they are the same:
print("qml.equal(m0==0, m0==1):", qml.equal(expr_eq_0, expr_eq_1))  # True (wrong)
print("qml.equal(m0==0, m0==0):", qml.equal(expr_eq_0, m0 == 0))  # True (correct control)

# --- Step 2: Wrap them in Conditional — the bug propagates ---
# "apply X when m0==0" vs "apply X when m0==1": completely different circuit semantics
cond_op_0 = qml.ops.Conditional(m0 == 0, qml.X(1))
cond_op_1 = qml.ops.Conditional(m0 == 1, qml.X(1))

print("qml.equal(cond(m0==0,X), cond(m0==1,X)):", qml.equal(cond_op_0, cond_op_1))  # True (wrong)

# --- Step 3: Downstream consequence — set/dict deduplication ---
# Because Operator.__eq__ delegates to qml.equal, Python set treats them as duplicates:
op_set = {cond_op_0, cond_op_1}
print(f"set size: {len(op_set)}  (should be 2, got 1 — one operation silently dropped)")

In [ ]:
import pennylane.estimator as qre

print("explicit precision control")
control = qre.GQSPTimeEvolution(qre.RX(0.1, wires=0), 1.0, 1.0, 0.01)
print(qre.estimate(control))

print("default precision trigger")
op = qre.GQSPTimeEvolution(qre.RX(0.1, wires=0), 1.0, 1.0)
print("stored poly_approx_precision:", op.poly_approx_precision)
print(qre.estimate(op))

In [ ]:
import pennylane.estimator as qre

print("eta=2 control")
control = qre.FirstQuantization(1000, 2, omega=100.0)
print("lamb:", control.lamb)
print("gates:", control.gates)
print("qubits:", control.qubits)

print("eta=1 trigger")
trigger = qre.FirstQuantization(1000, 1, omega=100.0)
print("lamb:", trigger.lamb)
print("gates:", trigger.gates)
print("qubits:", trigger.qubits)

In [ ]:
import pennylane.estimator as qre

one = qre.Identity()
two = qre.Identity(wires=[0, 1])
three = qre.Identity(wires=[0, 1, 2])

one_rep = one.resource_rep_from_op()
two_rep = two.resource_rep_from_op()
three_rep = three.resource_rep_from_op()

print("one instance num_wires:", one.num_wires)
print("one rep num_wires:", one_rep.num_wires)

print("two instance num_wires:", two.num_wires)
print("two rep num_wires:", two_rep.num_wires)

print("three instance num_wires:", three.num_wires)
print("three rep num_wires:", three_rep.num_wires)

print("two_rep == three_rep:", two_rep == three_rep)

In [ ]:
dev = qml.device("default.mixed", wires=2, readout_prob=0.1)


@qml.qnode(dev)
def circuit():
    return qml.expval(qml.PauliZ(0))


circuit()

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
import os

param = pnp.tensor(0.5, requires_grad=True)
print(f"Before: requires_grad={param.requires_grad}")  # True

op = qml.RX(param, wires=0)
dataset = qml.data.Dataset(op=op)

filepath = "/tmp/test_grad_roundtrip.h5"
dataset.write(filepath)

loaded = qml.data.Dataset()
loaded.read(filepath)

loaded_param = loaded.op.data[0]
print(
    f"After:  requires_grad={getattr(loaded_param, 'requires_grad', False)}"
)  # False — metadata lost

os.remove(filepath)

In [ ]:
import time
from unittest.mock import patch

import pennylane as qml

calls = []


def fake_download(dataset_url, dest, attributes, block_size, force, pbar_task):
    start = time.perf_counter()
    time.sleep(0.2)
    end = time.perf_counter()
    calls.append((dataset_url, start, end))


with patch("pennylane.data.data_manager._download_dataset", fake_download):
    qml.data.load(
        "qchem",
        molname="H2",
        basis="full",
        bondlength=[0.5, 1.1],
        num_threads=50,
    )

points = [t for _, a, b in calls for t in (a, b)]
max_concurrent = max(sum(a <= t < b for _, a, b in calls) for t in points)

elapsed = max(b for _, _, b in calls) - min(a for _, a, _ in calls)

print("elapsed:", round(elapsed, 2))
print("max_concurrent:", max_concurrent)

In [ ]:
import pennylane as qml
from pennylane.concurrency.executors.native.serial import SerialExec


def square(x):
    return x * x


executor = SerialExec()
data = [1, 2, 3, 4, 5]

result = executor.map(square, *data)

In [ ]:
from pennylane import qnode, device, sample, BasisEmbedding, Incrementer
import numpy as np

wires = [0, 1, 2]
work_wires = [3, 4]
init_state = [0, 1, 0]
expected = [0, 1, 1]

dev = device("default.qubit", wires=wires + work_wires)


@qnode(dev, shots=1)
def increment(wires, init_state, work_wires=None):
    BasisEmbedding(init_state, wires)
    Incrementer(wires, work_wires)
    return sample()


result = increment(wires, init_state, work_wires)[0]
result[: len(wires)]

In [ ]:
from pennylane.ops.op_math import Controlled2
import pennylane as qp
import numpy as np


class CH2(Controlled2, override_signature=True):
    """A new CH."""

    wire_argnames = ("wires",)

    wire_sizes = (2,)

    def __init__(self, wires):
        super().__init__(qp.H(wires[1]), wires[0], override_init_args={"wires": wires})


op = CH2([0, 1])
gates = [qp.RY(-np.pi / 4, wires=1)]
assert op.diagonalizing_gates() == gates
print(qp.matrix(qp.RY(-np.pi / 4, wires=1) @ op @ qp.adjoint(qp.RY(-np.pi / 4, wires=1))))

In [ ]:
import numpy as np
import pennylane as qml


def get_density_matrix(device_name):
    dev = qml.device(device_name, wires=1)

    @qml.qnode(dev)
    def circuit():
        qml.Hadamard(0)
        qml.S(0)
        return qml.density_matrix(wires=[0])

    return np.asarray(circuit())


actual = get_density_matrix("default.clifford")
expected = get_density_matrix("default.qubit")

print("default.clifford:")
print(actual)

print("\ndefault.qubit:")
print(expected)

print("\nMatches default.qubit:", np.allclose(actual, expected))
print("Hermitian:", np.allclose(actual, actual.conj().T))
print("Trace:", np.trace(actual))

In [ ]:
import numpy as np
import pennylane as qml


def run(device_name, tableau=None):
    kwargs = {} if tableau is None else {"tableau": tableau}
    dev = qml.device(device_name, wires=2, **kwargs)

    @qml.qnode(dev)
    def circuit():
        # Prepare (|00> + |11>) / sqrt(2)
        qml.Hadamard(0)
        qml.CNOT(wires=[0, 1])

        # The probability of projecting the Bell state onto |00> is 0.5.
        return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

    result = circuit()
    return result, np.shape(result)


for name, tableau in [
    ("default.qubit", None),
    ("default.clifford", True),
    ("default.clifford", False),
]:
    result, shape = run(name, tableau)
    label = name if tableau is None else f"{name}(tableau={tableau})"

    print(f"{label}:")
    print("  result:", result)
    print("  shape: ", shape)

In [ ]:
import pennylane as qml

dev = qml.device("default.clifford", wires=1)


@qml.set_shots([10, 20])
@qml.qnode(dev)
def counts_circuit():
    qml.Hadamard(0)
    return qml.counts(wires=[0])


@qml.set_shots([10, 20])
@qml.qnode(dev)
def sample_circuit():
    qml.Hadamard(0)
    return qml.sample(wires=[0])


@qml.set_shots([10, 20])
@qml.qnode(dev)
def expval_circuit():
    qml.Hadamard(0)
    return qml.expval(qml.Z(0))


print(counts_circuit())
print(sample_circuit())
print(expval_circuit())

In [ ]:
from pennylane.workflow.interfaces.jax import dtype
from pennylane.typing import AbstractArray
import pennylane as qp

qp.typing.Wire[2]

In [ ]:
import pennylane as qp
from pennylane.wires import Wires

isinstance(Wires([0, 1]), qp.typing.Wire[2])

In [ ]:
import numpy as np

isinstance(np.array(0 + 1.2j), qp.typing.Complex)

In [ ]:
k = 6
j = 1

(k >> j) & 1

In [ ]:
import pennylane as qp
from pennylane.ops.op_math.controlled2 import ControlledOp2
from copy import copy

copy(ControlledOp2(qp.X(1), [0]))

In [ ]:
import pennylane as qp

qp.adjoint(qp.S)

In [ ]:
from pennylane.typing import Wire
from pennylane import resource_rep, adjoint
from pennylane.ops import Hadamard, Prod, S

resource_rep(Prod, resources={Hadamard: 1, adjoint(S(Wire[1])): 1})

In [ ]:
from pennylane import QubitUnitary, qnode, device, state, add_decomps, register_resources
from pennylane.core.operator import Operation
import numpy as np


class MyQubitUnitary(Operation):

    def __init__(self, wires):
        super().__init__(wires)


def _my_resources(wires=None):
    return {resource_rep(QubitUnitary, num_wires=1): 1}


@register_resources(_my_resources)
def _my_unitary_decomp(wires):
    R = (
        np.array(
            [
                [1 + 2j, -3 + 4j],
                [3 + 4j, 1 - 2j],
            ]
        )
        * 30**-0.5
    )
    QubitUnitary(R, wires=wires)


add_decomps(MyQubitUnitary, _my_unitary_decomp)


@qnode(device("default.qubit", wires=[0]))
def circ():
    MyQubitUnitary([0])
    return state()


circ()

In [ ]:
import scipy
import pennylane as qp
import numpy as np

m = 2
M = 2**m

xmax = np.pi  # bound to region [-pi, pi]
xs = np.linspace(-xmax, xmax, M)

probs = np.array([scipy.stats.norm().pdf(x) for x in xs])
probs /= np.sum(probs)


def func(i):
    return np.sin(xs[i]) ** 2


n = 3
N = 2**n

target_wires = range(m + 1)
estimation_wires = range(m + 1, n + m + 1)


@qp.qnode(qp.device("default.qubit", wires=n + m + 1), shots=None)
def circuit():
    qp.QuantumMonteCarlo(
        probs,
        func,
        target_wires=target_wires,
        estimation_wires=estimation_wires,
    )
    return qp.probs(estimation_wires)


circuit()

In [ ]:
import pennylane as qp
import numpy as np

base = qp.PauliZ(0)
op = qp.pow(base=base, z=0.5)

eigvals = op.eigvals()

"""Expected eigenvalues for Z**0.5 are [1.0, 1.0j]"""
expected_eigvals = [1.0, 1j]
# assert np.allclose(eigvals, expected_eigvals)
eigvals

In [ ]:
base_eigvals = np.array([-1.0])
[complex(value) ** 0.5 for value in base_eigvals]

In [ ]:
import pennylane as qp
from pennylane.ops.op_math.pow2 import Pow2

sqrt_x = Pow2(qp.X(0), 0.5)
sqrt_x.decomposition()

In [ ]:
from pennylane import math
from pennylane.typing import Wire

math.shape(Wire[1])

In [ ]:
import pennylane as qp

op = qp.BasisRotation([[0, 1], [1, 0]], wires=[0, 1])
print(op.ndim_params)

In [ ]:
from pennylane.typing import _AbstractTypeFactory

Complex = _AbstractTypeFactory(complex)
f"{Complex=}".split("=")[0]

In [ ]:
import pennylane as qp
import numpy as np


@qp.qjit(capture=True, target="mlir")
@qp.qnode(qp.device("lightning.qubit", wires=[0, 1]))
def circuit():
    qp.BasisRotation(np.array([[0, 1], [1, 0]]), wires=[0, 1])
    return qp.state()


print(circuit.mlir)

In [ ]:
import pennylane as qp
import numpy as np

qp.math.shape(qp.typing.Wire[5])

In [ ]:
import pennylane as qp

qp.math.int_to_binary(-2, width=3)

In [ ]:
len(None)

In [ ]:
import pennylane as qp
import numpy as np
from pennylane.ops import X
from pennylane.ops.op_math.pow2 import Pow2
from pennylane.ops import QubitUnitary

Pow2(QubitUnitary(np.array([[0, 1], [1, 0]]), wires=[0]), 0.5).has_matrix

In [ ]:
import pennylane as qp
import numpy as np

dev = qp.device("default.qubit", wires=range(7))


@qp.qnode(dev)
def circuit():
    qp.SumOfSlatersPrep(
        np.ones(3) / np.sqrt(3),
        wires=[0, 1, 2],
        indices=(0, 1, 2),
        enumeration_wires=[3, 4],
        qrom_work_wires=[5],
        mcx_cache_wires=[6],
    )
    return qp.probs(wires=[0, 1, 2])


with qp.decomposition.toggle_graph_ctx(True):
    circuit()

In [ ]:
import pennylane as qp
from pennylane.data.attributes import DatasetPyTree

dst = DatasetPyTree(qp.RZ(0.5, wires=0))
dst.get_value()

In [ ]:
from pennylane import math

math.ones((1, 1))

In [ ]:
import numpy as np

np.eye(2) * (not 1)

In [ ]:
wires = [0, 1]

bitstring = [1, 0]

dimension = 2 ** len(wires)
bit_mask = sum(int(bit) << (len(wires) - index - 1) for index, bit in enumerate(bitstring))

basis_states = np.arange(0, dimension, 1, dtype=int)
flipped_states = basis_states ^ bit_mask

print(
    sparse.csr_matrix(
        (np.ones(dimension, dtype=int), (flipped_states, basis_states)),
        shape=(dimension, dimension),
    )
)

In [ ]:
from jax import numpy as jnp
import pennylane as qp

from pennylane.ops.op_math.controlled_ops import controlled_two_qubit_unitary_rule

kwargs = {
    "wires": jnp.array([0, 1, 2]),
    "U": jnp.eye(4),
    "control_values": jnp.array([False], dtype=bool),
    "work_wires": jnp.array([]),
    "work_wire_type": "borrowed",
}


@qp.qjit(capture=True, target="mlir", collect_decomp_rules=False)
@qp.qnode(device=qp.device("null.qubit", wires=4))
def c():
    controlled_two_qubit_unitary_rule(**kwargs)
    return qp.probs()


print(c.mlir)

In [ ]:
from pennylane.capture import run_autograph
from jax import numpy as jnp, make_jaxpr
from pennylane import capture

from pennylane.ops.op_math.controlled_ops import controlled_two_qubit_unitary_rule

capture.enable()

kwargs = {
    "wires": jnp.array([0, 1, 2]),
    "U": jnp.eye(4),
    "control_values": jnp.array([False], dtype=bool),
    "work_wires": jnp.array([]),
    "work_wire_type": "borrowed",
}

make_jaxpr(controlled_two_qubit_unitary_rule, static_argnums=(4,))(
    kwargs["U"],
    kwargs["wires"],
    kwargs["control_values"],
    kwargs["work_wires"],
    kwargs["work_wire_type"],
)

In [ ]:
import pennylane as qp
from jax import numpy as jnp
from pennylane.ops.op_math.controlled_ops import ctrl_decomp_bisect_rule

kwargs = {
    "wires": jnp.array([0, 1, 2, 3]),
    "U": jnp.eye(2),
}


@qp.qjit(capture=True, target="mlir", collect_decomp_rules=False)
@qp.qnode(device=qp.device("null.qubit", wires=[0, 1, 2, 3]))
def c():
    ctrl_decomp_bisect_rule(**kwargs)
    return qp.probs()


print(c.mlir)

In [ ]:
from catalyst.passes import graph_decomposition
from pennylane import qjit, qnode
import pennylane as qp
import numpy as np

enumeration_wires, identification_wires, qrom_work_wires, mcx_cache_wires = (
    [7, 8, 9],
    [10, 11, 12, 13, 14],
    [15, 16],
    [17, 18, 19, 20],
)

coefficients = np.arange(1, 8) / np.linalg.norm(np.arange(1, 8))
wires = list(range(7))
indices = (32, 16, 8, 4, 2, 1, 0)


@qjit(capture=True)
@graph_decomposition(
    gate_set={
        "TemporaryAnd",
        "QROM",
        "PauliX",
        "MultiplexerStatePreparation",
        "CNOT",
        "Adjoint(TemporaryAnd)",
    }
)
@qnode(qp.device("null.qubit"))
def circuit():
    qp.SumOfSlatersPrep(
        coefficients,
        wires,
        indices=indices,
        enumeration_wires=enumeration_wires,
        identification_wires=identification_wires,
        qrom_work_wires=qrom_work_wires,
        mcx_cache_wires=mcx_cache_wires,
    )


# circuit()
print(circuit.mlir)

In [ ]:
from pennylane.ops.op_math.controlled_ops import controlled_two_qubit_unitary_rule
from jax import numpy as jnp

kwargs = {
    "wires": jnp.array([0, 1, 2]),
    "U": jnp.eye(4),
    "control_values": jnp.array([False], dtype=bool),
    "work_wires": jnp.array([]),
    "work_wire_type": "borrowed",
}


@qp.qjit(capture=True, target="mlir", collect_decomp_rules=False)
@qp.qnode(device=qp.device("null.qubit", wires=4))
def c():
    controlled_two_qubit_unitary_rule(**kwargs)
    return qp.probs()


print(c.mlir)

In [ ]:
import pennylane as qp
import jax.numpy as jnp

qp.capture.enable()

dev = qp.device("null.qubit", wires=4)
U = 1 / jnp.sqrt(2) * jnp.array([[1, 1], [1, -1]], dtype=jnp.complex128)


@qp.qnode(dev)
def circuit():
    qp.ControlledQubitUnitary(U, control_values=[1], wires=[1, 0])
    return qp.probs()


pl_specs = qp.specs(circuit, level="device")()
print(pl_specs)

In [ ]:
import pennylane as qp
import jax.numpy as jnp

dev = qp.device("null.qubit", wires=4)
U = 1 / jnp.sqrt(2) * jnp.array([[1, 1], [1, -1]], dtype=jnp.complex128)


@qp.qjit(capture=True)
@qp.qnode(dev)
def circuit():
    qp.ControlledQubitUnitary(U, control_values=[1], wires=[1, 0])
    return qp.probs()


# Get the specs/resource usage
print(qp.specs(circuit, level="all-mlir")())

In [ ]:
import pennylane as qp
import catalyst

qp.decomposition.enable_graph()

gate_set = {"RZ", "S", "Adjoint(S)", "Hadamard", "CNOT", "GlobalPhase"}
fixed = {qp.RY: qp.list_decomps(qp.RY)["_ry_to_rz_cliff"]}  # a rule that already ships with PL


@qp.qjit(capture=True, target="mlir")
@qp.transforms.decompose(gate_set=gate_set, fixed_decomps=fixed)
@qp.qnode(qp.device("null.qubit", wires=1))
def circuit():
    qp.RY(0.3, wires=0)
    return qp.probs()


print(qp.specs(circuit, level="all")())

In [ ]:
import pennylane as qp


def f(x):
    return x


qp.apply_hint(f, {"identity": True})

In [ ]:
import pennylane as qp


def f(x):
    return x


qp.hint(f, {"identity": True})

In [ ]:
dev = qp.device("null.qubit", wires=2)


@qp.qjit(capture=True)  # if not specified by default
@qp.qnode(dev)
def circuit(x):

    @qp.hint({"branch-prob": 0.4})
    def true_fn():
        qp.RX(x, wires=0)

    def elif_fn():
        qp.RZ(x, wires=0)

    def false_fn():
        qp.CNOT(wires=(0, 1))

    qp.cond(x > 2.7, true_fn, false_fn, ((x > 1.4, elif_fn),))()
    return qp.expval(qp.Z(0))


qp.specs(circuit, level=0)(0.5).resources.quantum_operations

In [ ]:
import pennylane as qp
import numpy as np

rng = np.random.default_rng(42)
L, M, N = 1, 2, 3  # two-body fragments, modes, modals


def random_orthogonal(dim):
    q, r = np.linalg.qr(rng.standard_normal((dim, dim)))
    return q * np.sign(np.diag(r))


hamiltonian = qp.CGFHamiltonian(
    core_tensors=rng.standard_normal((L + 1, M, M, N, N)),
    leaf_tensors=np.stack(
        [np.stack([random_orthogonal(N) for _ in range(M)]) for _ in range(L + 1)]
    ),
    nuc_constant=0.5,
)

registers = qp.registers({"hadamard": 1, "system": M * N})
gate_set = {"Hadamard", "BasisRotation", "RZ", "CNOT", "PhaseShift", "ForLoop"}


@qp.qjit(capture=True)
@qp.decompose(gate_set=gate_set)
@qp.qnode(qp.device("lightning.qubit", wires=2 * N))
def trotter_circuit():
    qp.H(registers["hadamard"])

    qp.ctrl(
        qp.TrotterCGF(
            evolution_time=1.0,
            num_trotter_steps=10,
            hamiltonian=hamiltonian,
            wires=registers["system"],
        ),
        control=registers["hadamard"],
    )

    return qp.expval(qp.X(registers["hadamard"]))


trotter_circuit()

In [ ]:
import pennylane as qp
import numpy as np

U = np.eye(2)


@qp.qjit(capture=True)
@qp.decompose(gate_set={"BasisRotation", "RZ"})
@qp.qnode(qp.device("lightning.qubit", wires=2))
def circuit():
    qp.BasisRotation(wires=[0, 1], unitary_matrix=U)
    return qp.expval(qp.Z(0))


qp.specs(circuit, level="all")()  # CompileError: BasisRotation op was not bufferized

In [ ]:
from pennylane import math
import numpy as np

p = 1
s = 3
c = 2
grot_mat = np.array([[p * c, s], [-p * s, c]])
abs_c, abs_s = math.abs(grot_mat[:, 0])
abs_c, abs_s

In [ ]:
from pennylane.core.operator.meta import OperatorRegistry

print(OperatorRegistry().registry)

In [ ]:
from pennylane.core import Operator2


class MyNamedOp(Operator2):
    static_argnames = ("label",)

    def __init__(self, label, wires):
        super().__init__(label, wires=wires)


from pennylane.core import get_op_class_by_name

get_op_class_by_name("MyNamedOp")("my-label", [0, 1])

In [ ]:
import pennylane as qp

dev = qp.device("default.qubit")


@qp.transforms._tape_decompose(max_expansion=1)
@qp.qnode(dev)
def circuit():
    qp.ControlledSequence(qp.RX(0.25, wires=3), control=[0, 1, 2])
    return qp.state()


print(qp.draw(circuit, wire_order=[0, 1, 2, 3])())

In [1]:
import pennylane as qp


@qp.qjit(autograph=True)
@qp.transforms.merge_rotations
@qp.set_shots(10)
@qp.qnode(qp.device("null.qubit", wires=10))
def f():

    for i in range(10):
        qp.RZ(0.1 * (i + 1), wires=i)
        qp.Barrier()
        qp.RZ(0.1 * (i + 1), wires=i + 1)

    return qp.sample()


print(qp.analyze(f, level="all")())

ld: warning: object file (/private/var/folders/4h/q9djqckx5691k28bsw08dps40000gq/T/fne7vwi5v/f.o) was built for newer 'macOS' version (28.0) than being linked (27.0)


Device: null.qubit
Device wires: 10
Shots: Shots(total=10)
Levels:
- 0: Before MLIR Passes
- 1: After merge-rotations
- 2: After Device Preprocessing

↓Metric         Level→ |  0 |  1 |  2
-------------------------------------
Quantum operations:    |
- Total                | 30 | 30 | 30
  - Barrier            | 10 | 10 | 10
  - RZ                 | 20 | 20 | 20
Measurement processes: |
- sample(all wires)    |  1 |  1 |  1
Total wires            | 10 | 10 | 10
